In [ ]:
"""Chargement des bibliothèques nécessaires"""
# Pour manipuler les données
import pandas as pd
import numpy as np

# Pour gérer les chemins de fichiers
from pathlib import Path

# Import des modèles de sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

# Importer mlflow
import mlflow


# Pour sauvegarder les modèles
import joblib

# Définition des constantes
RANDOM_STATE = 42

# Définition des chemins
base_dir = Path().resolve().parents[1]
path_data = base_dir / "Patricia-Promise-Immo" / "data"/"processed"
features_path = path_data / "features_immo_data.csv"


In [72]:
# Chargement des données
df = pd.read_csv(f"{features_path}")
df

,Valeur fonciere,Nombre de lots,Code type local,Nombre pieces principales,Surface terrain,population,densite,altitude_moyenne,altitude_minimale,latitude_mairie,longitude_mairie,surface_total,prixm2
0,105000.0,2,3.0,0.0,0.0,1211.0,49.0,253.0,222.0,46.157,5.146,64.01,1640.368692
1,105000.0,2,2.0,3.0,0.0,1211.0,49.0,253.0,222.0,46.157,5.146,64.01,1640.368692
2,309700.0,1,1.0,4.0,0.0,1916.0,95.0,286.0,225.0,45.920,4.882,81.46,3801.865946
3,565423.0,1,2.0,3.0,0.0,10137.0,300.0,810.0,464.0,46.358,6.136,85.82,6588.475880
4,110000.0,1,2.0,2.0,0.0,16295.0,1055.0,554.0,330.0,46.108,5.826,45.67,2408.583315
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87475,65000.0,1,2.0,1.0,0.0,504078.0,4269.0,148.0,115.0,43.605,1.444,34.38,1890.634090
87476,64500.0,1,2.0,1.0,0.0,504078.0,4269.0,148.0,115.0,43.605,1.444,21.91,2943.861251
87477,191450.0,1,2.0,3.0,0.0,504078.0,4269.0,148.0,115.0,43.605,1.444,59.73,3205.256990
87478,300000.0,2,2.0,2.0,0.0,504078.0,4269.0,148.0,115.0,43.605,1.444,59.23,5065.000844


In [73]:
# Sélection des colonnes permettant de faire des prédictions
y = df["Valeur fonciere"]
#features_column = ["Code postal", "Code departement", "Code commune", "Type local","Surface reelle bati", "Nombre pieces principales", "Surface terrain","prixm2" ]
X = df.drop(columns=["Valeur fonciere"]) 

In [74]:
# Séparation des caractéristiques et de la cible
# y = df[target_column].copy()
#x= df[features_column].copy()

In [75]:
# Déclaration des données en ensembles d'entraînement et de test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, shuffle=True)

In [76]:
"""# Préparation des transformations pour les colonnes numériques et catégorielles
# Données numériques
num_columns = [col for col in df.columns if col in X_train.columns]

# Données catégorielles
#cat_columns = [col for col in X_train.columns if col not in num_columns]"""

'# Préparation des transformations pour les colonnes numériques et catégorielles\n# Données numériques\nnum_columns = [col for col in df.columns if col in X_train.columns]\n\n# Données catégorielles\n#cat_columns = [col for col in X_train.columns if col not in num_columns]'

In [77]:
df.columns.tolist()

['Valeur fonciere',
 'Nombre de lots',
 'Code type local',
 'Nombre pieces principales',
 'Surface terrain',
 'population',
 'densite',
 'altitude_moyenne',
 'altitude_minimale',
 'latitude_mairie',
 'longitude_mairie',
 'surface_total',
 'prixm2']

In [78]:
"""# Données numériques
# Tranformation des données numériques
transfo_num = Pipeline(steps=[
    ('imputation', SimpleImputer(strategy='median')),
    ('scaling', StandardScaler())
])"""

"# Données numériques\n# Tranformation des données numériques\ntransfo_num = Pipeline(steps=[\n    ('imputation', SimpleImputer(strategy='median')),\n    ('scaling', StandardScaler())\n])"

In [79]:
"""# Données Catégorielles
# Tranformation des données catégorielles
transfo_cat = Pipeline(steps=[
    ('imputation', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])"""

"# Données Catégorielles\n# Tranformation des données catégorielles\ntransfo_cat = Pipeline(steps=[\n    ('imputation', SimpleImputer(strategy='most_frequent')),\n    ('onehot', OneHotEncoder(handle_unknown='ignore'))\n])"

In [80]:
"""# Combinaison des données catégorielles et numériques pour la transformation
# Définition du transformateur complet
transformer_cols = ColumnTransformer(
    transformers=[
        #("data_cat", transfo_cat, cat_columns),
        ("data_num", transfo_num, num_columns)
    ]
)"""

'# Combinaison des données catégorielles et numériques pour la transformation\n# Définition du transformateur complet\ntransformer_cols = ColumnTransformer(\n    transformers=[\n        #("data_cat", transfo_cat, cat_columns),\n        ("data_num", transfo_num, num_columns)\n    ]\n)'

In [81]:
# Déclaration du modèle pipeline
model = RandomForestRegressor(n_estimators=100, max_depth=None, min_samples_split=2, n_jobs=-1, random_state=RANDOM_STATE)

In [82]:
# Entrainement du modèle
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [83]:
# Faire la prédiction
prediction = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
rmse = mean_squared_error(y_test, prediction)
mape = mean_absolute_percentage_error(y_test, prediction)
print("RMSE €:", round(rmse,0), "| MAPE %:", round(100*mape,2))

RMSE €: 15768934.0 | MAPE %: 0.32


In [85]:
"""# Définir le validateur KFold
cv = KFold(n_splits=5, shuffle=True, random_state=42)"""

'# Définir le validateur KFold\ncv = KFold(n_splits=5, shuffle=True, random_state=42)'

In [86]:
# Evaluation du modele
mae  = mean_absolute_error(y_test, prediction)
rmse = mean_squared_error(y_test, prediction)
r2   = r2_score(y_test, prediction)

In [87]:
# Affichages des métrics
print("=== Scores du test ===")
print(f"Moyenne des erreurs absolues :{mae:,.0f}")
print(f"Moyenne carrée des erreurs :{rmse:,.0f}")
print(f"Coefficient de détyermination :{r2:,.3f}")

=== Scores du test ===
Moyenne des erreurs absolues :717
Moyenne carrée des erreurs :15,768,934
Coefficient de détyermination :0.999


In [88]:
# definir le chemin de sauvegarde du modèle
base_dir = Path().resolve().parents[1]
artefact_path = base_dir/"Patricia-Promise-Immo"/"artefacts"
if not artefact_path.exists():
    artefact_path.mkdir(parents = True)

print(artefact_path)

D:\ProjectFolderDevAI_2025-2026\Immo_project\Patricia-Promise-Immo\artefacts


In [89]:
model_path = artefact_path/"Modèle_prix_dvf.joblib"
# Sauvegarder le pipeline
joblib.dump("model",f"{model_path}_v1")
print(f"Sauvegarde du pipeline du modèle éffectuée avec succès ==> {model_path}")

Sauvegarde du pipeline du modèle éffectuée avec succès ==> D:\ProjectFolderDevAI_2025-2026\Immo_project\Patricia-Promise-Immo\artefacts\Modèle_prix_dvf.joblib


In [90]:
import sklearn
print(sklearn.__version__)

1.7.2


In [91]:
import sklearn, inspect, sklearn.metrics as m
print("sklearn version:", sklearn.__version__)
print("metrics module :", m.__file__)
print("mse signature  :", inspect.signature(m.mean_squared_error))
print("mse module     :", m.mean_squared_error.__module__)


sklearn version: 1.7.2
metrics module : d:\ProjectFolderDevAI_2025-2026\Immo_project\env\lib\site-packages\sklearn\metrics\__init__.py
mse signature  : (y_true, y_pred, *, sample_weight=None, multioutput='uniform_average')
mse module     : sklearn.metrics._regression


In [ ]:
import mlflow

# Start an MLflow run
with mlflow.start_run():
    # Log a parameter (key-value pair)
    mlflow.log_param("param1", 5)
    
    # Log a metric; metrics can be updated throughout the run
    mlflow.log_metric("foo", 1)
    
    # Log an artifact (output file)
    with open("output.txt", "w") as f:
        f.write("Hello, world!")
    mlflow.log_artifact("output.txt", "")